# Goal 3 — Consultas, atualizações e transações em MongoDB

Neste notebook executamos consultas, inserções, atualizações, eliminações, transações e testes com vários clientes ao mesmo tempo. Usamos as seis coleções carregadas no Goal 2.

Execute as células pela ordem. Se necessário, instale as bibliotecas no kernel com `%pip install pymongo`. A variável `mongo_uri` indica o servidor e `mongo_base` indica a base de dados.

Os dados de teste usam IDs próprios e são eliminados no fim. As transações T1 e T2 precisam de um servidor configurado como replica set ou de um cluster Atlas, com os dados do Goal 2 já carregados.

Uma instrução `assert` verifica uma condição. Quando a condição é falsa, a execução para com um erro. `print` apresenta os resultados. Cada documento é um dicionário Python e os resultados são guardados em listas.

In [1]:
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path
from threading import Barrier
from time import perf_counter

from csv import DictReader
from pymongo import MongoClient
from pymongo.errors import DuplicateKeyError

mongo_uri = "mongodb://localhost:27017/"
mongo_base = "movielens"
client = MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)
client.admin.command("ping")
db = client[mongo_base]

colecoes = {"movies", "genres", "movie_genres", "users", "ratings", "tags"}
assert colecoes <= set(db.list_collection_names()), "Execute primeiro o Goal 2."

# Permite abrir o notebook na raiz do projeto ou na pasta Goal3.
pasta_dados = Path("../data/processed")
if not pasta_dados.is_dir():
    pasta_dados = Path("data/processed")

print("Base de dados:", db.name)

Base de dados: movielens


## Q1 — Consulta simples
Procuramos filmes lançados após 2010. `find` seleciona os documentos, `$gt` significa “maior que” e a projeção escolhe os campos apresentados. `_id: 0` retira o identificador automático do resultado. Ordenamos pelo movieId.

Mostramos os primeiros dez documentos. A lista resultado_q1 guarda todos os resultados.

In [2]:
resultado_q1 = list(
    db.movies.find(
        {"year": {"$gt": 2010}},
        {"_id": 0, "movieId": 1, "title": 1, "year": 1}
    ).sort("movieId", 1)
)
print("Número de resultados:", len(resultado_q1))
for documento in resultado_q1[:10]:
    print(documento)

Número de resultados: 1688
{'movieId': 82152, 'title': 'Beastly', 'year': 2011}
{'movieId': 83349, 'title': 'Green Hornet, The', 'year': 2011}
{'movieId': 83480, 'title': 'Season of the Witch', 'year': 2011}
{'movieId': 83613, 'title': 'Cowboys & Aliens', 'year': 2011}
{'movieId': 83910, 'title': 'Dilemma, The', 'year': 2011}
{'movieId': 84152, 'title': 'Limitless', 'year': 2011}
{'movieId': 84273, 'title': 'Zeitgeist: Moving Forward', 'year': 2011}
{'movieId': 84374, 'title': 'No Strings Attached', 'year': 2011}
{'movieId': 84392, 'title': 'Lincoln Lawyer, The', 'year': 2011}
{'movieId': 84414, 'title': 'All-Star Superman', 'year': 2011}


## Q2 — Consulta básica
Procuramos os 20 filmes mais recentes com ano conhecido. `$ne: None` exclui anos vazios. Na ordenação, `-1` significa decrescente e `1` significa crescente. Quando dois filmes têm o mesmo ano, o movieId decide a ordem. `limit(20)` limita o resultado a vinte filmes.

In [3]:
resultado_q2 = list(
    db.movies.find(
        {"year": {"$ne": None}},
        {"_id": 0, "movieId": 1, "title": 1, "year": 1}
    ).sort([("year", -1), ("movieId", 1)]).limit(20)
)
print("Número de resultados:", len(resultado_q2))
for documento in resultado_q2:
    print(documento)

Número de resultados: 20
{'movieId': 999980, 'title': 'Concurrency Test Movie', 'year': 2026}
{'movieId': 9999801, 'title': 'Concurrency Test Movie', 'year': 2026}
{'movieId': 9999912, 'title': 'Transaction Test Movie', 'year': 2026}
{'movieId': 99999123, 'title': 'Transaction Test Movie', 'year': 2026}
{'movieId': 122912, 'title': 'Avengers: Infinity War - Part I', 'year': 2018}
{'movieId': 182715, 'title': 'Annihilation', 'year': 2018}
{'movieId': 183011, 'title': 'The Commuter', 'year': 2018}
{'movieId': 183295, 'title': 'Insidious: The Last Key', 'year': 2018}
{'movieId': 183611, 'title': 'Game Night', 'year': 2018}
{'movieId': 183635, 'title': 'Maze Runner: The Death Cure', 'year': 2018}
{'movieId': 183897, 'title': 'Isle of Dogs', 'year': 2018}
{'movieId': 183911, 'title': 'The Clapper', 'year': 2018}
{'movieId': 183959, 'title': 'Tom Segura: Disgraceful', 'year': 2018}
{'movieId': 184015, 'title': 'When We First Met', 'year': 2018}
{'movieId': 184253, 'title': 'The Cloverfield P

## Q3 — Consulta intermédia
Calculamos a média e o número de avaliações de cada filme avaliado. A lista `q3` contém as etapas executadas por `aggregate`.

- `$group` reúne as avaliações do mesmo movieId.
- `$avg` calcula a média e `$sum: 1` conta as avaliações.
- `$lookup` procura o filme correspondente na coleção movies.
- `$unwind` transforma a lista devolvida pelo lookup num documento por filme.
- `$project` escolhe os campos finais e `$sort` ordena os resultados.

A média é calculada em decimal. A expressão floor(média × 10000 + 0.5) / 10000 arredonda para quatro casas decimais, com a mesma regra usada na consulta MySQL para valores positivos. O resultado final é convertido em número decimal Python.

Mostramos os primeiros dez documentos. A lista resultado_q3 guarda todos os resultados.

In [4]:
q3 = [
    # Agrupar as avaliações por filme.
    {"$group": {
        "_id": "$movieId",
        "averageRating": {"$avg": {"$toDecimal": "$rating"}},
        "totalRatings": {"$sum": 1}
    }},
    # Juntar os dados do filme.
    {"$lookup": {
        "from": "movies", "localField": "_id",
        "foreignField": "movieId", "as": "movie"
    }},
    {"$unwind": "$movie"},
    # Escolher os campos e arredondar a média.
    {"$project": {
        "_id": 0, "movieId": "$_id", "title": "$movie.title",
        "averageRating": {"$toDouble": {"$divide": [
            {"$floor": {"$add": [
                {"$multiply": ["$averageRating", 10000]},
                {"$toDecimal": "0.5"}
            ]}}, 10000
        ]}},
        "totalRatings": 1
    }},
    {"$sort": {"movieId": 1}}
]
resultado_q3 = list(db.ratings.aggregate(q3))
print("Número de resultados:", len(resultado_q3))
for documento in resultado_q3[:10]:
    print(documento)

Número de resultados: 9696
{'totalRatings': 214, 'movieId': 1, 'title': 'Toy Story', 'averageRating': 3.9206}
{'totalRatings': 109, 'movieId': 2, 'title': 'Jumanji', 'averageRating': 3.4312}
{'totalRatings': 52, 'movieId': 3, 'title': 'Grumpier Old Men', 'averageRating': 3.2596}
{'totalRatings': 7, 'movieId': 4, 'title': 'Waiting to Exhale', 'averageRating': 2.3571}
{'totalRatings': 49, 'movieId': 5, 'title': 'Father of the Bride Part II', 'averageRating': 3.0714}
{'totalRatings': 101, 'movieId': 6, 'title': 'Heat', 'averageRating': 3.9406}
{'totalRatings': 54, 'movieId': 7, 'title': 'Sabrina', 'averageRating': 3.1852}
{'totalRatings': 8, 'movieId': 8, 'title': 'Tom and Huck', 'averageRating': 2.875}
{'totalRatings': 16, 'movieId': 9, 'title': 'Sudden Death', 'averageRating': 3.125}
{'totalRatings': 132, 'movieId': 10, 'title': 'GoldenEye', 'averageRating': 3.4962}


## Q4 — Consulta avançada
Procuramos os dez filmes de Comedy com melhor média e pelo menos cem avaliações. A lista q4 contém todas as etapas da consulta.

Primeiro agrupamos e filtramos as avaliações. Depois juntamos os filmes e os géneros com lookup. No fim, escolhemos os campos, ordenamos e limitamos o resultado a dez filmes.

In [5]:
# Etapas da consulta Q4, executadas pela ordem indicada.
q4 = [
    # Contar as avaliações e calcular a média por filme.
    {"$group": {
        "_id": "$movieId",
        "averageRating": {"$avg": {"$toDecimal": "$rating"}},
        "totalRatings": {"$sum": 1}
    }},
    {"$match": {"totalRatings": {"$gte": 100}}},
    
    # Procurar o título do filme.
    {"$lookup": {
        "from": "movies", "localField": "_id",
        "foreignField": "movieId", "as": "movie"
    }},
    {"$unwind": "$movie"},
    
    # Procurar os géneros associados ao filme.
    {"$lookup": {
        "from": "movie_genres", "localField": "_id",
        "foreignField": "movieId", "as": "ligacao"
    }},
    {"$unwind": "$ligacao"},
    {"$lookup": {
        "from": "genres", "localField": "ligacao.genreId",
        "foreignField": "genreId", "as": "genre"
    }},
    {"$unwind": "$genre"},
    {"$match": {"genre.genre": "Comedy"}},
    
    # Apresentar os mesmos campos e arredondamento de Q3.
    {"$project": {
        "_id": 0, "movieId": "$_id", "title": "$movie.title",
        "averageRating": {"$toDouble": {"$divide": [
            {"$floor": {"$add": [
                {"$multiply": ["$averageRating", 10000]},
                {"$toDecimal": "0.5"}
            ]}}, 10000
        ]}},
        "totalRatings": 1
    }},
    {"$sort": {"averageRating": -1, "movieId": 1}},
    {"$limit": 10}
]
resultado_q4 = list(db.ratings.aggregate(q4))
print("Número de resultados:", len(resultado_q4))
for doc in resultado_q4:
    print(doc)

Número de resultados: 10
{'totalRatings': 139, 'movieId': 1197, 'title': 'Princess Bride, The', 'averageRating': 4.2266}
{'totalRatings': 303, 'movieId': 296, 'title': 'Pulp Fiction', 'averageRating': 4.1914}
{'totalRatings': 119, 'movieId': 4973, 'title': "Amelie (Fabuleux destin d'Amélie Poulain, Le)", 'averageRating': 4.1849}
{'totalRatings': 134, 'movieId': 1136, 'title': 'Monty Python and the Holy Grail', 'averageRating': 4.1716}
{'totalRatings': 329, 'movieId': 356, 'title': 'Forrest Gump', 'averageRating': 4.1641}
{'totalRatings': 181, 'movieId': 608, 'title': 'Fargo', 'averageRating': 4.116}
{'totalRatings': 102, 'movieId': 778, 'title': 'Trainspotting', 'averageRating': 4.0392}
{'totalRatings': 169, 'movieId': 1270, 'title': 'Back to the Future', 'averageRating': 4.0355}
{'totalRatings': 139, 'movieId': 6377, 'title': 'Finding Nemo', 'averageRating': 3.964}
{'totalRatings': 141, 'movieId': 1265, 'title': 'Groundhog Day', 'averageRating': 3.9433}


## Operações CRUD
Criamos um utilizador, um filme e uma avaliação temporários. Depois modificamos o título e a nota e eliminamos os três documentos.

Primeiro confirmamos que os IDs de teste estão livres. Escrevemos os campos de cada documento na própria operação. imdbId é uma string, de acordo com a validação do Goal 2.

- insert_one insere um documento.
- update_one altera um documento e $set define o novo valor de um campo.
- delete_one elimina um documento.
- find_one procura um documento pelo filtro indicado.

Depois de cada operação, imprimimos ou verificamos o resultado.

In [6]:
testUserId = 9999991
testMovieId = 9999991

def verificar_ids(ids):
    for i in ids:
        assert db.users.find_one({"userId": i}) is None, f"Utilizador {i} já existe."
        assert db.movies.find_one({"$or": [
            {"movieId": i}, {"imdbId": i}, {"imdbId": str(i)}, {"tmdbId": i}
        ]}) is None, f"Identificador de filme {i} já existe."

verificar_ids([testUserId])

### I1 — Inserção de um utilizador

In [7]:
db.users.insert_one({"userId": testUserId, "longitude": 40.0, "latitude": -8.0})
assert db.users.count_documents({"userId": testUserId}) == 1
print(db.users.find_one({"userId": testUserId}, {"_id": 0}))

{'userId': 9999991, 'longitude': 40.0, 'latitude': -8.0}


### I2 — Inserção de um filme

In [8]:
db.movies.insert_one({
    "movieId": testMovieId,
    "title": "Test Movie",
    "year": 2026,
    "imdbId": str(testMovieId),
    "tmdbId": testMovieId
})
assert db.movies.count_documents({"movieId": testMovieId}) == 1
print(db.movies.find_one({"movieId": testMovieId}, {"_id": 0}))

{'movieId': 9999991, 'title': 'Test Movie', 'year': 2026, 'imdbId': '9999991', 'tmdbId': 9999991}


### I3 — Inserção de uma avaliação

In [9]:
filtro_teste = {"userId": testUserId, "movieId": testMovieId}
db.ratings.insert_one({
    **filtro_teste, "rating": 3.5, "timestamp": datetime.now(timezone.utc)
})
assert db.ratings.find_one(filtro_teste)["rating"] == 3.5
print(db.ratings.find_one(filtro_teste, {"_id": 0}))

{'userId': 9999991, 'movieId': 9999991, 'rating': 3.5, 'timestamp': datetime.datetime(2026, 11, 26, 15, 39, 50, 86000)}


### U1 — Atualização do título

In [10]:
db.movies.update_one({"movieId": testMovieId}, {"$set": {"title": "Test Movie Updated"}})
assert db.movies.find_one({"movieId": testMovieId})["title"] == "Test Movie Updated"
print(db.movies.find_one({"movieId": testMovieId}, {"_id": 0}))

{'movieId': 9999991, 'title': 'Test Movie Updated', 'year': 2026, 'imdbId': '9999991', 'tmdbId': 9999991}


### U2 — Atualização da avaliação

In [11]:
db.ratings.update_one(filtro_teste, {"$set": {"rating": 4.5}})
assert db.ratings.find_one(filtro_teste)["rating"] == 4.5
print(db.ratings.find_one(filtro_teste, {"_id": 0}))

{'userId': 9999991, 'movieId': 9999991, 'rating': 4.5, 'timestamp': datetime.datetime(2026, 11, 26, 15, 39, 50, 86000)}


### D1 — Eliminação da avaliação

In [12]:
db.ratings.delete_one(filtro_teste)
assert db.ratings.count_documents(filtro_teste) == 0
print("Avaliação eliminada.")

Avaliação eliminada.


### D2 — Eliminação do filme

In [13]:
db.movies.delete_one({"movieId": testMovieId})
assert db.movies.count_documents({"movieId": testMovieId}) == 0
print("Filme eliminado.")

Filme eliminado.


### D3 — Eliminação do utilizador

In [14]:
db.users.delete_one({"userId": testUserId})
assert db.users.count_documents({"userId": testUserId}) == 0
print("Utilizador eliminado.")

Utilizador eliminado.


## Transações
Uma transação agrupa várias alterações. Em T1, inserimos um utilizador, um filme e uma avaliação. Em T2, repetimos uma avaliação para provocar um erro.

`start_session` cria a sessão. `start_transaction` inicia a transação. `commit_transaction` confirma as alterações. `abort_transaction` anula as alterações em caso de erro. Cada escrita recebe `session=session` para participar nessa transação.

O índice único de userId e movieId, criado no Goal 2, impede duas avaliações do mesmo utilizador para o mesmo filme. `estado(i)` conta os documentos de teste nas três coleções.

Para executar transações localmente:
1. Inicie o servidor com uma pasta de dados própria e a opção `--replSet rs0`.
2. Ligue o mongosh a essa instância e execute `rs.initiate()` uma vez.
3. Configure o mesmo endereço no Goal 2 e neste notebook e carregue os dados nessa instância.

Um replica set com um membro permite fazer estes testes no computador.

No Windows deste projeto, execute configurar_replica_set.ps1 num PowerShell como Administrador para configurar o servico existente e verificar as transacoes. Depois reinicie o kernel para renovar a ligacao.

In [15]:
hello = client.admin.command("hello")
if "setName" not in hello and hello.get("msg") != "isdbgrid":
    raise RuntimeError(
        "T1 e T2 requerem um replica set ou Atlas. "
        "No Windows, execute configurar_replica_set.ps1 como Administrador e reinicie o kernel."
    )

assert any(
    indice.get("unique") and [campo for campo, ordem in indice["key"]] == ["userId", "movieId"]
    for indice in db.ratings.index_information().values()
), "Execute o Goal 2: falta o índice único (userId, movieId)."

def estado(i):
    return {
        "users": db.users.count_documents({"userId": i}),
        "movies": db.movies.count_documents({"movieId": i}),
        "ratings": db.ratings.count_documents({"userId": i, "movieId": i})
    }

### T1 — Transação com sucesso (COMMIT)

In [16]:
i = 999991

db.users.delete_one({"userId": i})
db.movies.delete_one({"movieId": i})
db.ratings.delete_many({"userId": i})

verificar_ids([i])

with client.start_session() as session:
    session.start_transaction()
    try:
        # Inserir as três alterações na mesma transação.
        db.users.insert_one({"userId": i, "longitude": 40.0, "latitude": -8.0}, session=session)
        
        db.movies.insert_one({
            "movieId": i, "title": "Transaction Test Movie",
            "year": 2026, "imdbId": str(i), "tmdbId": i
        }, session=session)
        
        db.ratings.insert_one({
            "userId": i, "movieId": i, "rating": 4.5,
            "timestamp": datetime.now(timezone.utc)
        }, session=session)

        session.commit_transaction()
        print("COMMIT executado com sucesso.")
    except Exception:
        session.abort_transaction()
        raise

print(estado(i))
assert estado(i) == {"users": 1, "movies": 1, "ratings": 1}

COMMIT executado com sucesso.
{'users': 1, 'movies': 1, 'ratings': 1}


### T2 — Transação com falha e ROLLBACK
A segunda avaliação repete a combinação userId e movieId. O MongoDB devolve DuplicateKeyError. abort_transaction anula também o utilizador, o filme e a primeira avaliação. Confirmamos que as três contagens são zero.

In [17]:
i = 999992
verificar_ids([i])

with client.start_session() as session:
    session.start_transaction()
    
    try:
        db.users.insert_one({"userId": i, "longitude": 40.0, "latitude": -8.0}, session=session)
        
        db.movies.insert_one({
            "movieId": i, "title": "Rollback Test Movie",
            "year": 2026, "imdbId": str(i), "tmdbId": i
        }, session=session)
        
        db.ratings.insert_one({
            "userId": i, "movieId": i, "rating": 3.5,
            "timestamp": datetime.now(timezone.utc)
        }, session=session)

        # Repetir a chave para provocar uma falha.
        db.ratings.insert_one({
            "userId": i, "movieId": i, "rating": 4.5,
            "timestamp": datetime.now(timezone.utc)
        }, session=session)
    
    except DuplicateKeyError as erro:
        session.abort_transaction()
        print("ROLLBACK executado. Código do erro:", erro.code)
        print("Mensagem:", erro.details.get("errmsg", str(erro)))
    
    except Exception:
        session.abort_transaction()
        raise
    
    else:
        session.abort_transaction()
        raise AssertionError("Era esperado um erro de chave duplicada.")

print(estado(i))
assert estado(i) == {"users": 0, "movies": 0, "ratings": 0}

ROLLBACK executado. Código do erro: 11000
Mensagem: E11000 duplicate key error collection: movielens.ratings index: userId_1_movieId_1 dup key: { userId: 999992, movieId: 999992 }
{'users': 0, 'movies': 0, 'ratings': 0}


### Remoção dos dados de T1

In [18]:
db.ratings.delete_one({"userId": 999991, "movieId": 999991})
db.movies.delete_one({"movieId": 999991})
db.users.delete_one({"userId": 999991})
assert estado(999991) == {"users": 0, "movies": 0, "ratings": 0}

## Concorrência
Mantemos quatro cenários: leituras, escritas em avaliações diferentes, atualização perdida e incremento atómico. C1 e C2 usam 1, 5 e 10 clientes. C3 usa três clientes.

A função testar faz o trabalho comum:
1. Cria uma ligação para cada cliente.
2. Espera pelos restantes clientes com Barrier.
3. Executa a operação e mede o tempo em milissegundos.
4. Imprime o tempo, o resultado e os erros de cada cliente.

Cada teste define apenas a operação a executar. A função recebe a coleção ratings, o ID do cliente e a barreira. A função resumir calcula os tempos e conta os sucessos e erros.

In [19]:
NUM_CLIENTES = [1, 5, 10]

def resumir(resultados, consistente):
    tempos = [r["Tempo (ms)"] for r in resultados if r["Sucesso"]]
    return {
        "N.º de clientes": len(resultados),
        "Tempo médio (ms)": sum(tempos) / len(tempos) if tempos else None,
        "Tempo mínimo (ms)": min(tempos) if tempos else None,
        "Tempo máximo (ms)": max(tempos) if tempos else None,
        "Sucessos": sum(r["Sucesso"] for r in resultados),
        "Erros": sum(not r["Sucesso"] for r in resultados),
        "Consistência": consistente
    }

def testar(operacao, ids):
    barreira = Barrier(len(ids), timeout=30)

    def cliente(i):
        try:
            with MongoClient(mongo_uri, serverSelectionTimeoutMS=5000) as ligacao:
                ligacao.admin.command("ping")
                colecao = ligacao[mongo_base].ratings
                barreira.wait()

                inicio = perf_counter()
                resultado = operacao(colecao, i, barreira)
                tempo = (perf_counter() - inicio) * 1000

            return {"Cliente": i, "Tempo (ms)": round(tempo, 2),
                    "Resultado": resultado, "Sucesso": True, "Erro": None}
        except Exception as erro:
            barreira.abort()
            return {"Cliente": i, "Tempo (ms)": None,
                    "Resultado": None, "Sucesso": False, "Erro": str(erro)}

    with ThreadPoolExecutor(max_workers=len(ids)) as executor:
        resultados = list(executor.map(cliente, ids))

    for resultado in resultados:
        print(resultado)
    return resultados

### C1 — Leituras concorrentes
Cada cliente executa Q4 e compara os documentos com resultado_q4. Resultado igual a True confirma que recebeu os dados esperados.

In [20]:
def leitura(colecao, i, barreira):
    documentos = list(colecao.aggregate(q4))
    return documentos == resultado_q4

resultados_c1 = []
resumo_c1 = []
for n in NUM_CLIENTES:
    print("\nC1 — Clientes:", n)
    resultados = testar(leitura, list(range(1, n + 1)))
    consistente = all(r["Sucesso"] and r["Resultado"] is True for r in resultados)
    resultados_c1.extend(resultados)
    resumo_c1.append(resumir(resultados, consistente))
    assert consistente


C1 — Clientes: 1
{'Cliente': 1, 'Tempo (ms)': 326.21, 'Resultado': True, 'Sucesso': True, 'Erro': None}

C1 — Clientes: 5
{'Cliente': 1, 'Tempo (ms)': 420.87, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 2, 'Tempo (ms)': 382.86, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 3, 'Tempo (ms)': 340.04, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 4, 'Tempo (ms)': 449.17, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 5, 'Tempo (ms)': 411.18, 'Resultado': True, 'Sucesso': True, 'Erro': None}

C1 — Clientes: 10
{'Cliente': 1, 'Tempo (ms)': 435.29, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 2, 'Tempo (ms)': 444.38, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 3, 'Tempo (ms)': 453.31, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 4, 'Tempo (ms)': 427.41, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 5, 'Tempo (ms)': 423.21, 'Resultado': True, 'Sucesso': True, 'E

### C2 — Escritas concorrentes sem conflito
Criamos um filme, dez utilizadores e dez avaliações com nota 3.0. Cada cliente atualiza uma avaliação diferente para 4.5.

Antes de cada teste, `repor_dados_c2` volta a colocar as notas em 3.0. `verificar_c2(n)` confirma que as primeiras n avaliações passaram para 4.5 e que as restantes continuam em 3.0.

In [23]:
id_filme_c2 = 999980
ids_c2 = list(range(999970, 999980))

db.ratings.delete_many({
    "$or": [
        {"userId": {"$in": ids_c2}},
        {"movieId": id_filme_c2},
    ]
})
db.tags.delete_many({"userId": {"$in": ids_c2}})
db.users.delete_many({"userId": {"$in": ids_c2}})
db.movies.delete_one({"movieId": id_filme_c2})

verificar_ids(ids_c2 + [id_filme_c2])
db.movies.insert_one({
    "movieId": id_filme_c2, "title": "Concurrency Test Movie",
    "year": 2026, "imdbId": str(id_filme_c2), "tmdbId": id_filme_c2
})
db.users.insert_many([{"userId": i, "longitude": 40.0, "latitude": -8.0} for i in ids_c2])
db.ratings.insert_many([
    {"userId": i, "movieId": id_filme_c2, "rating": 3.0,
     "timestamp": datetime.now(timezone.utc)}
    for i in ids_c2
])

def repor_dados_c2():
    db.ratings.update_many({"movieId": id_filme_c2}, {"$set": {"rating": 3.0}})

def verificar_c2(n):
    obtido = [
        (doc["userId"], doc["rating"])
        for doc in db.ratings.find({"movieId": id_filme_c2}).sort("userId", 1)
    ]
    esperado = [(i, 4.5 if pos < n else 3.0) for pos, i in enumerate(ids_c2)]
    return obtido == esperado

In [24]:
def escrita(colecao, i, barreira):
    resultado = colecao.update_one(
        {"userId": i, "movieId": id_filme_c2},
        {"$set": {"rating": 4.5}}
    )
    assert resultado.matched_count == 1, "Avaliação não encontrada."
    return True

resultados_c2 = []
resumo_c2 = []
for n in NUM_CLIENTES:
    repor_dados_c2()
    print("\nC2 — Clientes:", n)
    resultados = testar(escrita, ids_c2[:n])
    consistente = verificar_c2(n)
    resultados_c2.extend(resultados)
    resumo_c2.append(resumir(resultados, consistente))
    assert consistente
    assert all(r["Sucesso"] for r in resultados)


C2 — Clientes: 1
{'Cliente': 999970, 'Tempo (ms)': 4.06, 'Resultado': True, 'Sucesso': True, 'Erro': None}

C2 — Clientes: 5
{'Cliente': 999970, 'Tempo (ms)': 4.06, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999971, 'Tempo (ms)': 5.49, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999972, 'Tempo (ms)': 4.41, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999973, 'Tempo (ms)': 5.35, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999974, 'Tempo (ms)': 5.37, 'Resultado': True, 'Sucesso': True, 'Erro': None}

C2 — Clientes: 10
{'Cliente': 999970, 'Tempo (ms)': 3.4, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999971, 'Tempo (ms)': 2.53, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999972, 'Tempo (ms)': 2.7, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999973, 'Tempo (ms)': 2.77, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 999974, 'Tempo (ms)': 2.06, 'Result

### C3.1 — Atualização perdida
Três clientes leem a mesma nota inicial, 3.0. A barreira garante que todos leem antes de começarem a escrever.

Cada cliente soma 0.5 ao valor que leu e escreve 3.5 com `$set`. O valor esperado após três incrementos seria 4.5, mas o valor observado neste teste é 3.5. As escritas substituem o resultado umas das outras e perdem-se dois incrementos.

Consistência igual a False mostra o problema que este teste pretende demonstrar.

In [25]:
id_user_c3 = ids_c2[0]
id_filme_c3 = id_filme_c2
filtro_c3 = {"userId": id_user_c3, "movieId": id_filme_c3}
valor_inicial = 3.0
incremento = 0.5
num_clientes_c3 = 3
valor_esperado = valor_inicial + incremento * num_clientes_c3

def sem_protecao(colecao, i, barreira):
    valor = colecao.find_one(filtro_c3)["rating"]
    barreira.wait()  # Todos leem antes de alguém escrever.
    colecao.update_one(filtro_c3, {"$set": {"rating": valor + incremento}})
    return valor + incremento

repor_dados_c2()
resultados_c31 = testar(sem_protecao, list(range(1, num_clientes_c3 + 1)))
valor_final_c31 = db.ratings.find_one(filtro_c3)["rating"]
consistencia_c31 = valor_final_c31 == valor_esperado

print("Esperado:", valor_esperado, "| Obtido:", valor_final_c31)
assert all(r["Sucesso"] for r in resultados_c31)
assert valor_final_c31 == valor_inicial + incremento

{'Cliente': 1, 'Tempo (ms)': 2.98, 'Resultado': 3.5, 'Sucesso': True, 'Erro': None}
{'Cliente': 2, 'Tempo (ms)': 3.22, 'Resultado': 3.5, 'Sucesso': True, 'Erro': None}
{'Cliente': 3, 'Tempo (ms)': 2.73, 'Resultado': 3.5, 'Sucesso': True, 'Erro': None}
Esperado: 4.5 | Obtido: 3.5


### C3.2 — Incremento atómico
Cada cliente executa update_one com $inc. O servidor soma 0.5 ao valor atual da avaliação. Os três incrementos devem produzir 4.5.

No MySQL usamos SELECT FOR UPDATE. Aqui usamos uma atualização atómica de um documento. Ao comparar os tempos, considere os mecanismos de cada operação. Em C3.1, o tempo também inclui a espera depois da leitura para reproduzir a atualização perdida.

In [26]:
def com_protecao(colecao, i, barreira):
    resultado = colecao.update_one(filtro_c3, {"$inc": {"rating": incremento}})
    assert resultado.matched_count == 1, "Avaliação não encontrada."
    return True

repor_dados_c2()
resultados_c32 = testar(com_protecao, list(range(1, num_clientes_c3 + 1)))
valor_final_c32 = db.ratings.find_one(filtro_c3)["rating"]
consistencia_c32 = valor_final_c32 == valor_esperado

print("Esperado:", valor_esperado, "| Obtido:", valor_final_c32)
assert all(r["Sucesso"] for r in resultados_c32)
assert consistencia_c32

{'Cliente': 1, 'Tempo (ms)': 4.47, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 2, 'Tempo (ms)': 4.52, 'Resultado': True, 'Sucesso': True, 'Erro': None}
{'Cliente': 3, 'Tempo (ms)': 5.62, 'Resultado': True, 'Sucesso': True, 'Erro': None}
Esperado: 4.5 | Obtido: 4.5


### Resumo dos testes
Os resumos reúnem os tempos, sucessos, erros e verificações desta execução. O segundo resumo mostra os valores inicial, esperado e final de C3.1 e C3.2.

Na discussão dos resultados, explique se os clientes receberam os mesmos dados, se todas as escritas foram realizadas e por que razão C3.1 termina em 3.5 e C3.2 em 4.5. Compare também estes resultados com os testes MySQL.

In [27]:
resumo_final = []
for resumo in resumo_c1:
    resumo_final.append({"Teste": "C1 — Leituras", **resumo})
for resumo in resumo_c2:
    resumo_final.append({"Teste": "C2 — Escritas sem conflito", **resumo})
resumo_final.append({
    "Teste": "C3.1 — Atualização perdida",
    **resumir(resultados_c31, consistencia_c31)
})
resumo_final.append({
    "Teste": "C3.2 — Incremento atómico",
    **resumir(resultados_c32, consistencia_c32)
})

for resumo in resumo_final:
    print(resumo)

print("\nComparação dos incrementos:")
print("Valor inicial:", valor_inicial)
print("Valor esperado:", valor_esperado)
print("Com $set:", valor_final_c31, "| Consistência:", consistencia_c31)
print("Com $inc:", valor_final_c32, "| Consistência:", consistencia_c32)

{'Teste': 'C1 — Leituras', 'N.º de clientes': 1, 'Tempo médio (ms)': 326.21, 'Tempo mínimo (ms)': 326.21, 'Tempo máximo (ms)': 326.21, 'Sucessos': 1, 'Erros': 0, 'Consistência': True}
{'Teste': 'C1 — Leituras', 'N.º de clientes': 5, 'Tempo médio (ms)': 400.824, 'Tempo mínimo (ms)': 340.04, 'Tempo máximo (ms)': 449.17, 'Sucessos': 5, 'Erros': 0, 'Consistência': True}
{'Teste': 'C1 — Leituras', 'N.º de clientes': 10, 'Tempo médio (ms)': 435.48099999999994, 'Tempo mínimo (ms)': 421.96, 'Tempo máximo (ms)': 453.31, 'Sucessos': 10, 'Erros': 0, 'Consistência': True}
{'Teste': 'C2 — Escritas sem conflito', 'N.º de clientes': 1, 'Tempo médio (ms)': 4.06, 'Tempo mínimo (ms)': 4.06, 'Tempo máximo (ms)': 4.06, 'Sucessos': 1, 'Erros': 0, 'Consistência': True}
{'Teste': 'C2 — Escritas sem conflito', 'N.º de clientes': 5, 'Tempo médio (ms)': 4.936, 'Tempo mínimo (ms)': 4.06, 'Tempo máximo (ms)': 5.49, 'Sucessos': 5, 'Erros': 0, 'Consistência': True}
{'Teste': 'C2 — Escritas sem conflito', 'N.º de cl

### Remoção dos dados de teste e encerramento
Eliminamos as avaliações, o filme e os utilizadores usados na concorrência. Os asserts confirmam que estes documentos foram removidos. Por fim, `client.close()` fecha a ligação.

In [28]:
db.ratings.delete_many({"userId": {"$in": ids_c2}, "movieId": id_filme_c2})
db.movies.delete_one({"movieId": id_filme_c2})
db.users.delete_many({"userId": {"$in": ids_c2}})

assert db.ratings.count_documents({"movieId": id_filme_c2}) == 0
assert db.movies.count_documents({"movieId": id_filme_c2}) == 0
assert db.users.count_documents({"userId": {"$in": ids_c2}}) == 0
client.close()
print("Dados de teste eliminados e ligação encerrada.")

Dados de teste eliminados e ligação encerrada.
